# 07 — Analytical Inverse Kinematics for a 2-Link Planar Arm

**Section:** Manipulation · **Mirrors MATLAB:** *Inverse Kinematics*

For a planar 2-link arm with link lengths $l_1$, $l_2$ and end-effector position $(x, y)$:

$$\cos\theta_2 = \frac{x^2 + y^2 - l_1^2 - l_2^2}{2 l_1 l_2}$$

$$\theta_1 = \arctan2(y, x) - \arctan2(l_2 \sin\theta_2,\ l_1 + l_2 \cos\theta_2)$$

The choice of $+\arccos$ vs $-\arccos$ for $\theta_2$ selects the **elbow-up** or **elbow-down** branch.


## Intuition — what's actually going on?

**Forward** kinematics is easy: if I tell you the angles of all the joints in a robot arm, you can compute where the hand ends up by stacking rotations. **Inverse** kinematics is the harder reverse problem: I tell you where I want the hand, and you have to figure out the joint angles.

For a 2-link planar arm there's a closed-form solution — you can write down explicit formulas for the two joint angles. There are usually **two valid solutions** ("elbow up" and "elbow down"), like how your arm can reach the same point with your elbow pointing forward or backward.

The trick is the **law of cosines**: the triangle formed by (shoulder, elbow, hand) has all three side lengths known (the two link lengths, plus the distance from shoulder to target). The law of cosines gives the elbow angle directly. Then a bit of trig gives the shoulder angle.

Real industrial robots have 6 or 7 joints in 3D, no closed form exists, and you need numerical IK (see notebook 16).


### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| $r^2 = x^2 + y^2$ | `r2 = x * x + y * y` |
| $\cos\theta_2 = \dfrac{r^2 - l_1^2 - l_2^2}{2 l_1 l_2}$ (clipped) | `c2 = np.clip((r2 - l1**2 - l2**2) / (2*l1*l2), -1, 1)` |
| $\theta_2 = \pm\arccos(\cos\theta_2)$ | `th2 = np.arccos(c2) if elbow_up else -np.arccos(c2)` |
| $\theta_1 = \arctan2(y, x) - \arctan2(l_2\sin\theta_2,\ l_1 + l_2\cos\theta_2)$ | `th1 = np.arctan2(y, x) - np.arctan2(l2*np.sin(th2), l1 + l2*np.cos(th2))` |
| Forward kinematics for visualization | `p1 = (l1*cos(th1), l1*sin(th1)); p2 = p1 + (l2*cos(th1+th2), l2*sin(th1+th2))` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

l1, l2 = 1.0, 1.0


def fk(theta):
    th1, th2 = theta
    p1 = np.array([l1 * np.cos(th1), l1 * np.sin(th1)])
    p2 = p1 + np.array([l2 * np.cos(th1 + th2), l2 * np.sin(th1 + th2)])
    return p1, p2


def ik(target, elbow_up=True):
    # Closed-form 2-link IK. det(J) = l1*l2*sin(th2); singular at th2 = 0, pi.
    x, y = target
    r2 = x * x + y * y
    if r2 < 1e-12:                   # council fix (pass 7, Cauchy)
        raise ValueError("target at base - theta1 undefined")
    c2 = np.clip((r2 - l1 ** 2 - l2 ** 2) / (2 * l1 * l2), -1, 1)
    th2 = np.arccos(c2) if elbow_up else -np.arccos(c2)
    th1 = np.arctan2(y, x) - np.arctan2(l2 * np.sin(th2), l1 + l2 * np.cos(th2))
    return np.array([th1, th2])


def ik_both_branches(target):
    # Council fix (pass 7, Asada): return BOTH elbow-up and elbow-down.
    return [ik(target, elbow_up=True), ik(target, elbow_up=False)]


In [ ]:
# Trace a circular trajectory
N = 60
phi = np.linspace(0, 2 * np.pi, N)
targets = np.column_stack([1.0 + 0.4 * np.cos(phi), 0.7 + 0.4 * np.sin(phi)])

fig, ax = plt.subplots(figsize=(8, 8))
for k, t in enumerate(targets):
    theta = ik(t)
    p1, p2 = fk(theta)
    a = 0.15 + 0.7 * (k / N)
    ax.plot([0, p1[0], p2[0]], [0, p1[1], p2[1]], 'b-', alpha=a, lw=1.2)
    ax.plot(p1[0], p1[1], 'ko', markersize=3, alpha=a)

ax.plot(targets[:, 0], targets[:, 1], 'r--', lw=1.5, label='Target circle')
ax.plot(0, 0, 'gs', markersize=12, label='Base')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1.0, 2.5)
ax.set_aspect('equal'); ax.grid(); ax.legend()
ax.set_title('2-Link IK: arm poses traced through a circle (elbow-up)')
plt.tight_layout()
plt.show()


## References & rigor notes

**Workspace.** Reachable end-effector positions form the annulus $|l_1 - l_2| \le r \le l_1 + l_2$. Inside there are exactly two IK solutions (elbow-up and elbow-down); on the inner or outer boundary they degenerate to one (a singular configuration where the arm is fully extended or folded). **For equal links** $l_1 = l_2$ (as in this notebook), the inner boundary collapses to a disk and $\theta_1$ becomes undefined at the origin (the arm folds onto itself and is free to rotate about the base).

**Singularities.** At $\theta_2 = 0$ or $\theta_2 = \pi$ the Jacobian is rank-deficient — the arm loses a degree of freedom. Closed-form IK still returns an answer, but small motions in joint space can cause large motions in task space (or vice versa).

**Complexity.** $O(1)$ — closed-form, dominated by a few trig evaluations. Compare to numerical IK (notebook 16) which is iterative.

**References.**
- Murray, R. M., Li, Z., & Sastry, S. S. (1994). *A Mathematical Introduction to Robotic Manipulation*, CRC Press, ch. 3.
- Spong, M. W., Hutchinson, S., & Vidyasagar, M. (2006). *Robot Modeling and Control*, Wiley, ch. 3.
- Lynch, K. M., & Park, F. C. (2017). *Modern Robotics*, Cambridge University Press, ch. 6.
